<a href="https://colab.research.google.com/github/akritib25/FoodSafety/blob/main/Google_Review_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 01_scraper.ipynb
# SCC Vermin Closure × Google Reviews Scraper
# Goal: collect 1-2 star reviews from last 6 months
#       for 15 restaurants (10 vermin + 5 control)
# Output: reviews_clean.csv
# ============================================================

# Cell 1 — Install
!pip install requests pandas -q
print("Ready")

Ready


In [ ]:
# Cell 2 — Setup
import requests, json, time, os
import pandas as pd

APIFY_KEY = os.environ.get('APIFY_KEY', '')
print(f"Apify: {APIFY_KEY[:8]}...")

Apify: apify_ap...


In [ ]:
# Cell 3 — Restaurant list
restaurants = [
    # Vermin closures
    {"name": "Bikaner Sweet",             "address": "1635 Hollenbeck Ave Sunnyvale CA",         "closure": "2026-06-04", "type": "vermin"},
    {"name": "Red Robin",                 "address": "2200 Eastridge Loop San Jose CA",           "closure": "2026-05-29", "type": "vermin"},
    {"name": "Pacific Catch",             "address": "1875 S Bascom Ave Campbell CA",             "closure": "2026-05-20", "type": "vermin"},
    {"name": "Black Bear Diner",          "address": "395 Leavesley Rd Gilroy CA",                "closure": "2026-05-12", "type": "vermin"},
    {"name": "Benihana",                  "address": "10123 N Wolfe Rd Cupertino CA",             "closure": "2026-04-01", "type": "vermin"},
    {"name": "The Counter",               "address": "3055 Olin Ave San Jose CA",                 "closure": "2026-03-13", "type": "vermin"},
    {"name": "Round Table Pizza",         "address": "18482 Prospect Rd Saratoga CA",             "closure": "2026-03-17", "type": "vermin"},
    {"name": "Wienerschnitzel",           "address": "1940 S Bascom Ave Campbell CA",             "closure": "2026-04-09", "type": "vermin"},
    {"name": "Emelina's",                "address": "460 E William St San Jose CA",              "closure": "2025-12-23", "type": "vermin"},
    {"name": "Tomi Sushi Seafood Buffet", "address": "2200 Eastridge Loop San Jose CA",           "closure": "2026-01-22", "type": "vermin"},
    # Control — closed for water/sewage not vermin
    {"name": "Crawfish LA",               "address": "2857 Senter Rd San Jose CA",                "closure": "2026-05-15", "type": "control"},
    {"name": "MJ Sushi",                  "address": "2305 El Camino Real Palo Alto CA",          "closure": "2026-04-30", "type": "control"},
    {"name": "Voyager Craft Coffee",      "address": "111 W St John St San Jose CA",              "closure": "2026-04-08", "type": "control"},
    {"name": "Lados",                     "address": "115 Plaza Dr Sunnyvale CA",                 "closure": "2026-04-30", "type": "control"},
    {"name": "Go Fish Poke Bar",          "address": "244 Stanford Shopping Center Palo Alto CA", "closure": "2026-03-02", "type": "control"},
]
print(f"Restaurants: {len(restaurants)} ({sum(1 for r in restaurants if r['type']=='vermin')} vermin, {sum(1 for r in restaurants if r['type']=='control')} control)")

Restaurants: 15 (10 vermin, 5 control)


In [ ]:
# Cell 4 — Start all 15 scrape jobs simultaneously
def start_scrape(name, address, api_key, max_reviews=50):
    r = requests.post(
        "https://api.apify.com/v2/acts/compass~google-maps-reviews-scraper/runs",
        json={
            "searchStringsArray": [f"{name} {address}"],
            "maxReviewsPerQuery": max_reviews,
            "reviewsSort":        "newest",
            "language":           "en"
        },
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type":  "application/json"
        },
        timeout=30
    )
    if r.status_code in [200, 201]:
        run_id = r.json()['data']['id']
        print(f"  ✓ {name[:40]:40s} → {run_id}")
        return run_id
    print(f"  ✗ {name[:40]:40s} → {r.status_code}")
    return None

print("Starting all 15 scrape jobs...")
run_ids = []
for rest in restaurants:
    run_id = start_scrape(rest['name'], rest['address'], APIFY_KEY)
    run_ids.append(run_id)
    time.sleep(0.5)

successful = sum(1 for r in run_ids if r)
print(f"\nStarted: {successful}/{len(restaurants)}")
print("Waiting 3 minutes for all jobs to complete...")
time.sleep(180)

Starting all 15 scrape jobs...
  ✗ Bikaner Sweet                            → 403
  ✗ Red Robin                                → 403
  ✗ Pacific Catch                            → 403
  ✗ Black Bear Diner                         → 403
  ✗ Benihana                                 → 403
  ✗ The Counter                              → 403
  ✗ Round Table Pizza                        → 403
  ✗ Wienerschnitzel                          → 403
  ✗ Emelina's                                → 403
  ✗ Tomi Sushi Seafood Buffet                → 403
  ✗ Crawfish LA                              → 403
  ✗ MJ Sushi                                 → 403
  ✗ Voyager Craft Coffee                     → 403
  ✗ Lados                                    → 403
  ✗ Go Fish Poke Bar                         → 403

Started: 0/15
Waiting 3 minutes for all jobs to complete...


In [ ]:
import requests, json, time, os
import pandas as pd

APIFY_KEY = os.environ.get('APIFY_KEY', '')

# Paste your run IDs from earlier
run_ids = [
    "kRHBI3xTD229k6iG4",  # Bikaner Sweet
    "aQpecZsNwpv8QZCXS",  # Red Robin
    "mscpGaLY4sLdtfUvo",  # Pacific Catch
    "Oo8FYDi3NSn2V2yfb",  # Black Bear Diner
    "bzDTRQULSYVl7gbZ0",  # Benihana
    "Asfx1nLxrvkqQVGWF",  # The Counter
    "3EfPwdLWJANdSFz69",  # Round Table Pizza
    "32ogZj7JxwHHc7Ej2",  # Wienerschnitzel
]

restaurants = [
    {"name": "Bikaner Sweet",     "address": "1635 Hollenbeck Ave Sunnyvale CA",    "closure": "2026-06-04", "type": "vermin"},
    {"name": "Red Robin",         "address": "2200 Eastridge Loop San Jose CA",      "closure": "2026-05-29", "type": "vermin"},
    {"name": "Pacific Catch",     "address": "1875 S Bascom Ave Campbell CA",        "closure": "2026-05-20", "type": "vermin"},
    {"name": "Black Bear Diner",  "address": "395 Leavesley Rd Gilroy CA",           "closure": "2026-05-12", "type": "vermin"},
    {"name": "Benihana",          "address": "10123 N Wolfe Rd Cupertino CA",        "closure": "2026-04-01", "type": "vermin"},
    {"name": "The Counter",       "address": "3055 Olin Ave San Jose CA",            "closure": "2026-03-13", "type": "vermin"},
    {"name": "Round Table Pizza", "address": "18482 Prospect Rd Saratoga CA",        "closure": "2026-03-17", "type": "vermin"},
    {"name": "Wienerschnitzel",   "address": "1940 S Bascom Ave Campbell CA",        "closure": "2026-04-09", "type": "vermin"},
]

def get_results(run_id, api_key):
    if not run_id:
        return []
    headers = {"Authorization": f"Bearer {api_key}"}
    r = requests.get(
        f"https://api.apify.com/v2/actor-runs/{run_id}",
        headers=headers, timeout=30
    )
    data       = r.json().get('data', {})
    status     = data.get('status', 'UNKNOWN')
    dataset_id = data.get('defaultDatasetId', '')
    print(f"    status={status}")
    if not dataset_id:
        return []
    r2 = requests.get(
        f"https://api.apify.com/v2/datasets/{dataset_id}/items",
        headers=headers, timeout=30
    )
    return r2.json()

def start_scrape(name, address, api_key, max_reviews=50):
    r = requests.post(
        "https://api.apify.com/v2/acts/compass~google-maps-reviews-scraper/runs",
        json={
            "searchStringsArray": [f"{name} {address}"],
            "maxReviewsPerQuery": max_reviews,
            "reviewsSort":        "newest",
            "language":           "en"
        },
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type":  "application/json"
        },
        timeout=30
    )
    if r.status_code in [200, 201]:
        run_id = r.json()['data']['id']
        print(f"  ✓ {name[:40]:40s} → {run_id}")
        return run_id
    print(f"  ✗ {name[:40]:40s} → {r.status_code}")
    return None

print("All functions restored")
print(f"Apify key: {APIFY_KEY[:8]}...")

All functions restored
Apify key: apify_ap...


In [ ]:
# Collect the 8 completed jobs
print("Collecting 8 completed jobs...")
all_rows = []

for rest, run_id in zip(restaurants, run_ids):
    print(f"  {rest['name'][:35]}...")
    results = get_results(run_id, APIFY_KEY)
    count   = 0
    for item in results:
        text = item.get('text', '') or ''
        pub  = item.get('publishedAtDate', '') or ''
        if not text.strip() or not pub:
            continue
        rev_date    = pd.to_datetime(pub[:10])
        closure     = pd.to_datetime(rest['closure'])
        days_before = (closure - rev_date).days
        rating      = item.get('stars') or item.get('rating')
        all_rows.append({
            'restaurant':   rest['name'],
            'closure_date': rest['closure'],
            'type':         rest['type'],
            'source':       'google',
            'review_date':  rev_date.strftime('%Y-%m-%d'),
            'days_before':  days_before,
            'rating':       rating,
            'text':         text.strip(),
            'author':       item.get('name', ''),
        })
        count += 1
    print(f"    → {count} reviews collected")

df_raw = pd.DataFrame(all_rows)
print(f"\nTotal reviews: {len(df_raw)}")
print(df_raw.groupby('restaurant')['rating'].count())

  Bikaner Sweet...
    status=SUCCEEDED
    → 32 reviews collected
  Red Robin...
    status=SUCCEEDED
    → 757 reviews collected
  Pacific Catch...
    status=SUCCEEDED
    → 1030 reviews collected
  Black Bear Diner...
    status=ABORTED
    → 806 reviews collected
  Benihana...
    status=ABORTED
    → 968 reviews collected
  The Counter...
    status=SUCCEEDED
    → 654 reviews collected
  Round Table Pizza...
    status=SUCCEEDED
    → 63 reviews collected
  Wienerschnitzel...
    status=SUCCEEDED
    → 241 reviews collected

Total reviews: 4551
restaurant
Benihana              968
Bikaner Sweet          32
Black Bear Diner      806
Pacific Catch        1030
Red Robin             757
Round Table Pizza      63
The Counter           654
Wienerschnitzel       241
Name: rating, dtype: int64


In [ ]:
# Start remaining 7 with delay to avoid 402
remaining = [
    {"name": "Emelina's",                "address": "460 E William St San Jose CA",              "closure": "2025-12-23", "type": "vermin"},
    {"name": "Tomi Sushi Seafood Buffet", "address": "2200 Eastridge Loop San Jose CA",           "closure": "2026-01-22", "type": "vermin"},
    {"name": "Crawfish LA",               "address": "2857 Senter Rd San Jose CA",                "closure": "2026-05-15", "type": "control"},
    {"name": "MJ Sushi",                  "address": "2305 El Camino Real Palo Alto CA",          "closure": "2026-04-30", "type": "control"},
    {"name": "Voyager Craft Coffee",      "address": "111 W St John St San Jose CA",              "closure": "2026-04-08", "type": "control"},
    {"name": "Lados",                     "address": "115 Plaza Dr Sunnyvale CA",                 "closure": "2026-04-30", "type": "control"},
    {"name": "Go Fish Poke Bar",          "address": "244 Stanford Shopping Center Palo Alto CA", "closure": "2026-03-02", "type": "control"},
]

print("Starting remaining 7 with 5 second delay between each...")
run_ids_2 = []
for rest in remaining:
    run_id = start_scrape(rest['name'], rest['address'], APIFY_KEY)
    run_ids_2.append(run_id)
    time.sleep(5)

successful = sum(1 for r in run_ids_2 if r)
print(f"\nStarted: {successful}/{len(remaining)}")
print("Waiting 3 minutes...")
time.sleep(180)

Starting remaining 7 with 5 second delay between each...
  ✗ Emelina's                                → 403
  ✗ Tomi Sushi Seafood Buffet                → 403
  ✗ Crawfish LA                              → 403
  ✗ MJ Sushi                                 → 403
  ✗ Voyager Craft Coffee                     → 403
  ✗ Lados                                    → 403
  ✗ Go Fish Poke Bar                         → 403

Started: 0/7
Waiting 3 minutes...


In [ ]:
# Clean and filter the 4551 reviews we already have
df_clean = df_raw[
    (df_raw['days_before'] > 0) &
    (df_raw['days_before'] <= 180) &
    (df_raw['rating'].isin([1, 2])) &
    (df_raw['text'].str.len() > 20)
].copy().reset_index(drop=True)

print(f"Raw reviews:    {len(df_raw)}")
print(f"After cleaning: {len(df_clean)}")
print(f"\nBy restaurant:")
print(df_clean.groupby(['restaurant','type'])[['rating']].count().to_string())
print(f"\nRating split:")
print(df_clean['rating'].value_counts().sort_index())
print(f"\nDays before closure range: {df_clean['days_before'].min()} → {df_clean['days_before'].max()}")

Raw reviews:    4551
After cleaning: 78

By restaurant:
                          rating
restaurant        type          
Benihana          vermin      52
Bikaner Sweet     vermin       2
Black Bear Diner  vermin       2
Pacific Catch     vermin       4
Red Robin         vermin       6
Round Table Pizza vermin       1
The Counter       vermin       6
Wienerschnitzel   vermin       5

Rating split:
rating
1    63
2    15
Name: count, dtype: int64

Days before closure range: 2 → 164


In [ ]:
# Collect remaining 7
print("Collecting remaining 7...")

for rest, run_id in zip(remaining, run_ids_2):
    if not run_id:
        print(f"  ✗ {rest['name'][:35]} → skipped")
        continue
    print(f"  {rest['name'][:35]}...")
    results = get_results(run_id, APIFY_KEY)
    count   = 0
    for item in results:
        text = item.get('text', '') or ''
        pub  = item.get('publishedAtDate', '') or ''
        if not text.strip() or not pub:
            continue
        rev_date    = pd.to_datetime(pub[:10])
        closure     = pd.to_datetime(rest['closure'])
        days_before = (closure - rev_date).days
        rating      = item.get('stars') or item.get('rating')
        all_rows.append({
            'restaurant':   rest['name'],
            'closure_date': rest['closure'],
            'type':         rest['type'],
            'source':       'google',
            'review_date':  rev_date.strftime('%Y-%m-%d'),
            'days_before':  days_before,
            'rating':       rating,
            'text':         text.strip(),
            'author':       item.get('name', ''),
        })
        count += 1
    print(f"    → {count} reviews")

# Rebuild full dataframe
df_raw = pd.DataFrame(all_rows)
df_clean = df_raw[
    (df_raw['days_before'] > 0) &
    (df_raw['days_before'] <= 180) &
    (df_raw['rating'].isin([1, 2])) &
    (df_raw['text'].str.len() > 20)
].copy().reset_index(drop=True)

print(f"\nTotal raw:     {len(df_raw)}")
print(f"Total cleaned: {len(df_clean)}")
print(f"\nBy restaurant and type:")
print(df_clean.groupby(['restaurant','type'])['rating'].count().to_string())

# Save
df_raw.to_csv('reviews_raw.csv', index=False)
df_clean.to_csv('reviews_clean.csv', index=False)
print(f"\n✓ Saved reviews_raw.csv")
print(f"✓ Saved reviews_clean.csv")
print(f"\nReady for Notebook 2 — analyser")

  ✗ Emelina's → skipped
  ✗ Tomi Sushi Seafood Buffet → skipped
  ✗ Crawfish LA → skipped
  ✗ MJ Sushi → skipped
  ✗ Voyager Craft Coffee → skipped
  ✗ Lados → skipped
  ✗ Go Fish Poke Bar → skipped

Total raw:     4551
Total cleaned: 78

By restaurant and type:
restaurant         type  
Benihana           vermin    52
Bikaner Sweet      vermin     2
Black Bear Diner   vermin     2
Pacific Catch      vermin     4
Red Robin          vermin     6
Round Table Pizza  vermin     1
The Counter        vermin     6
Wienerschnitzel    vermin     5

✓ Saved reviews_raw.csv
✓ Saved reviews_clean.csv

Ready for Notebook 2 — analyser


In [ ]:
# Save what we have and move on
df_raw.to_csv('reviews_raw.csv', index=False)
df_clean.to_csv('reviews_clean.csv', index=False)

print("=== SCRAPER COMPLETE (PARTIAL) ===")
print(f"Restaurants scraped: 8/15")
print(f"Raw reviews:         {len(df_raw)}")
print(f"Clean reviews:       {len(df_clean)}")
print(f"\nBy restaurant:")
print(df_clean.groupby(['restaurant','type'])['rating'].count().to_string())
print(f"\nSaved reviews_clean.csv — ready for analyser notebook")
print(f"\nRemaining 7 restaurants → scrape tomorrow when Apify resets")

=== SCRAPER COMPLETE (PARTIAL) ===
Restaurants scraped: 8/15
Raw reviews:         4551
Clean reviews:       78

By restaurant:
restaurant         type  
Benihana           vermin    52
Bikaner Sweet      vermin     2
Black Bear Diner   vermin     2
Pacific Catch      vermin     4
Red Robin          vermin     6
Round Table Pizza  vermin     1
The Counter        vermin     6
Wienerschnitzel    vermin     5

Saved reviews_clean.csv — ready for analyser notebook

Remaining 7 restaurants → scrape tomorrow when Apify resets


In [ ]:
# Show all 78 clean reviews for manual inspection
pd.set_option('display.max_colwidth', 200)

print(f"=== ALL CLEAN REVIEWS ({len(df_clean)} total) ===\n")

for rest_name in df_clean['restaurant'].unique():
    subset = df_clean[df_clean['restaurant'] == rest_name].sort_values('days_before')
    print(f"\n{'='*60}")
    print(f"RESTAURANT: {rest_name}")
    closure = subset['closure_date'].iloc[0]
    rtype   = subset['type'].iloc[0]
    print(f"Closure date: {closure} | Type: {rtype}")
    print(f"Reviews (1-2 star, last 6 months): {len(subset)}")
    print(f"{'='*60}")
    for _, row in subset.iterrows():
        print(f"\n  Date: {row['review_date']} | {row['days_before']} days before closure | ★{row['rating']}")
        print(f"  {row['text'][:300]}")
        print(f"  {'-'*50}")

=== ALL CLEAN REVIEWS (78 total) ===


RESTAURANT: Bikaner Sweet
Closure date: 2026-06-04 | Type: vermin
Reviews (1-2 star, last 6 months): 2

  Date: 2026-05-22 | 13 days before closure | ★1
  Service is literally non existent here. Avoid this restaurant as much as possible.
  --------------------------------------------------

  Date: 2026-02-22 | 102 days before closure | ★1
  I was very disappointed with my recent visit to *Bikanos Indian Restaurant* The namkeen snacks & sweets were not good at all, it tasted oily and lacked freshness. It seemed like namkeen snacks were fried in overused oil, as there was a unpleasant aftertaste.

Food quality and freshness are basic exp
  --------------------------------------------------

RESTAURANT: Red Robin
Closure date: 2026-05-29 | Type: vermin
Reviews (1-2 star, last 6 months): 6

  Date: 2026-04-05 | 54 days before closure | ★2
  I ordered the Whiskey Chicken wrap and got really no sauce, very dry. They took awhile to take our order, first